## Narrative Position

**Pipeline step:** Feature 구성과 measurement validity (4/6) — **우리 팀 차별점 layer 2 (전반)**

**This notebook answers:** *"토큰을 어떤 수치 feature로 바꾸고, 그 수치가 무엇을 측정한다고 가정하는가?"*

**Next:** `03b_expanded_dictionary_fasttext.ipynb` (fastText 확장 사전) → `04_feature_validation_v2.ipynb`

**Linked robustness evidence:** `../jiwoo/02_seed_dictionary_validation.ipynb` — seed별 개별 Spearman ρ

**Evaluation rubric coverage:** Feature 종류 (seed TF-IDF / 차원별 TF-IDF) · Feature 해석

**Why this matters for our team's framing:** Feature는 "숫자"가 아니라 *"무엇을 측정한다고 우리가 주장하는 변환"*이다. 각 feature가 verbosity와 어떻게 다른지, 그리고 그 차이가 KCGS 등급과 어떻게 연결되는지를 다음 두 단계에서 검증한다. fastText 기반 expanded dictionary 흐름은 `03b`에서 별도로 정리한다.

---


# 03 · Feature Build v2 — 토큰을 firm-year 수치로

**Input**  : `data/04_preprocessed/tokenized_{exp_id}.csv`  
**Output** : `data/05_features/features_{exp_id}.parquet`  
           + `data/05_features/feature_build_log.csv`

---

## 이 단계가 답하는 한 줄

> "ESG 어휘 강도를 어떻게 firm-year 1행짜리 수치로 변환했는가?"

형태소 분석된 토큰 리스트(joined_text)를 받아 4종 feature를 계산한다:

| Feature | 정의 | cheap-talk 관련성 |
|---|---|---|
| `g_signal_ratio` | G_SIGNAL 토큰 수 / total_tokens | PRIMARY — 거버넌스 어휘 밀도 |
| `esg_signal_ratio` | ESG_SIGNAL 토큰 수 / total_tokens | E/S 어휘 밀도 |
| `esg_tfidf_concentration` | TF-IDF top-200 중 seed 어휘 mass 비율 | 변별적 ESG 신호 강도 |
| `bp_contamination_rate` | BOILERPLATE 토큰 비율 | 잡음 진단 |

**왜 비율(ratio)인가:** 보고서 길이가 기업마다 다르다. 단순 카운트는  
긴 보고서를 쓰는 기업이 항상 높은 값을 갖는다(verbosity bias).  
비율은 이를 완화하지만 완전히 제거하지는 않는다 → `log_tokens` 통제 필수.

---
**실험 ID** : `exp_B` (pre-reg baseline) / `exp_E` / `exp_F` (primary)  
**Random seed** : 42

In [ ]:
import sys, os
from pathlib import Path

# project root를 sys.path에 추가
PROJECT_ROOT = Path.cwd().parent.parent
sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
import warnings
warnings.filterwarnings('ignore')

from pipeline_config import (
    D_PREPROC, D_FEATURES, D_MERGED, O_TABLES,
    RANDOM_SEED, ID_COLS, SEED_30, SEED_DICT,
    G_SIGNAL_SET, ESG_SIGNAL_SET, BOILERPLATE_SET,
    GRADE_TO_7, GRADE_TO_4,
    ROBUSTNESS_EXP, PRIMARY_EXP,
)

np.random.seed(RANDOM_SEED)
print(f"PROJECT_ROOT : {PROJECT_ROOT}")
print(f"D_FEATURES   : {D_FEATURES}")

## 1. 입력 데이터 로드 — tokenized_{exp_id}.csv

In [ ]:
def load_tokenized(exp_id: str, preproc_dir: Path) -> pd.DataFrame:
    """tokenized_{exp_id}.csv 로드 + 기본 검증"""
    path = preproc_dir / f"tokenized_{exp_id}.csv"
    assert path.exists(), f"파일 없음: {path}"
    
    df = pd.read_csv(path, dtype={"stock_code": str, "corp_code": str,
                                   "rcept_no": str, "fiscal_year": int,
                                   "esg_year": int})
    df["stock_code"] = df["stock_code"].str.zfill(6)
    df["joined_text"] = df["joined_text"].fillna("")
    
    print(f"[{exp_id}] shape: {df.shape}")
    print(f"  unique stock_code: {df['stock_code'].nunique()}")
    print(f"  fiscal_year dist: {df['fiscal_year'].value_counts().sort_index().to_dict()}")
    print(f"  empty joined_text: {(df['joined_text'].str.len() == 0).sum()}")
    return df

# 모든 exp 로드
tokenized_dfs = {}
for exp_id in ROBUSTNESS_EXP:
    try:
        tokenized_dfs[exp_id] = load_tokenized(exp_id, D_PREPROC)
    except AssertionError as e:
        print(f"[SKIP] {e}")
        
print(f"\n로드 완료: {list(tokenized_dfs.keys())}")

## 2. Feature 계산 함수

### 2.1 g_signal_ratio / esg_signal_ratio / bp_contamination_rate

In [ ]:
def compute_ratio_features(joined_text: str) -> dict:
    """
    joined_text (공백 분리 토큰)에서 ratio feature 계산.
    
    주의: ESG_SIGNAL_SET과 G_SIGNAL_SET이 일부 겹침.
    겹치는 토큰은 각 set에서 독립적으로 카운트 (중복 허용).
    이는 '어휘 비율'이 중복을 허용해야 각 차원의 독립적 밀도를
    측정할 수 있기 때문이다.
    """
    tokens = joined_text.split() if joined_text else []
    total = len(tokens)
    
    if total == 0:
        return {
            "total_tokens": 0, "log_tokens": None,
            "esg_signal_count": 0, "g_signal_count": 0, "bp_count": 0,
            "esg_signal_ratio": None, "g_signal_ratio": None,
            "bp_contamination_rate": None, "esg_g_relative": None,
        }
    
    esg_n = sum(1 for t in tokens if t in ESG_SIGNAL_SET)
    g_n   = sum(1 for t in tokens if t in G_SIGNAL_SET)
    bp_n  = sum(1 for t in tokens if t in BOILERPLATE_SET)
    denom = esg_n + g_n + bp_n
    
    return {
        "total_tokens":          total,
        "log_tokens":            round(np.log(total + 1), 4),
        "esg_signal_count":      esg_n,
        "g_signal_count":        g_n,
        "bp_count":              bp_n,
        "esg_signal_ratio":      round(esg_n / total, 5),
        "g_signal_ratio":        round(g_n   / total, 5),
        "bp_contamination_rate": round(bp_n  / total, 5),
        "esg_g_relative":        round((esg_n + g_n) / denom, 4) if denom > 0 else None,
    }

### 2.2 esg_tfidf_concentration

**정의:** corpus 전체 TF-IDF matrix에서 각 문서의 top-K token 중  
seed 어휘(SEED_30)가 차지하는 mass 비율.

**해석:** 높음 → 이 기업의 ESG 어휘가 corpus에서 변별적으로 분포  
낮음 → ESG 어휘가 general language 속에 희석 (cheap-talk 패턴)

In [ ]:
def compute_tfidf_concentration(
    documents: list,
    seed_words: list = None,
    top_k: int = 200,
    min_df: int = 2,
) -> list:
    """
    각 document의 TF-IDF top-K 토큰 중 seed_words mass 비율.
    
    Decision: word-level TF-IDF (형태소 분석 후 joined_text 기준).
    Alternative: char n-gram — 형태소 분석 없이 쓸 수 있지만
                 seed 단어 검출이 subword 단위로 깨짐 → 사용 안 함.
    """
    if seed_words is None:
        seed_words = SEED_30
    
    if len(documents) < 2:
        return [None] * len(documents)
    
    vec = TfidfVectorizer(
        tokenizer=lambda x: x.split(),
        lowercase=False,
        min_df=min_df,
        token_pattern=None,   # tokenizer 직접 지정 시 필요
    )
    
    try:
        tfidf_mat = vec.fit_transform(documents)
    except Exception as e:
        print(f"TF-IDF 오류: {e}")
        return [None] * len(documents)
    
    feature_names = np.array(vec.get_feature_names_out())
    seed_mask = np.isin(feature_names, seed_words)
    
    concentrations = []
    for i in range(tfidf_mat.shape[0]):
        row = tfidf_mat[i].toarray().flatten()
        if row.sum() == 0:
            concentrations.append(None)
            continue
        
        top_k_idx    = np.argsort(row)[-top_k:]
        top_k_mass   = row[top_k_idx].sum()
        seed_mass    = row[top_k_idx][seed_mask[top_k_idx]].sum()
        
        concentrations.append(
            round(seed_mass / top_k_mass, 5) if top_k_mass > 0 else None
        )
    
    return concentrations

### 2.3 Seed TF-IDF (E/S/G 차원별)

In [ ]:
def compute_seed_tfidf_by_dim(
    documents: list,
    seed_dict: dict = None,
    min_df: int = 2,
) -> dict:
    """
    E/S/G 차원별 seed TF-IDF sum을 firm-year별로 계산.
    
    Returns: {'seed_tfidf_E': list, 'seed_tfidf_S': list, 'seed_tfidf_G': list}
    """
    if seed_dict is None:
        seed_dict = SEED_DICT
    
    vec = TfidfVectorizer(
        tokenizer=lambda x: x.split(),
        lowercase=False,
        min_df=min_df,
        token_pattern=None,
    )
    
    try:
        tfidf_mat = vec.fit_transform(documents)
        feature_names = np.array(vec.get_feature_names_out())
    except Exception as e:
        print(f"TF-IDF 오류: {e}")
        empty = [None] * len(documents)
        return {"seed_tfidf_E": empty, "seed_tfidf_S": empty, "seed_tfidf_G": empty}
    
    result = {}
    for dim, seeds in seed_dict.items():
        seed_mask = np.isin(feature_names, seeds)
        scores = []
        for i in range(tfidf_mat.shape[0]):
            row = tfidf_mat[i].toarray().flatten()
            scores.append(round(float(row[seed_mask].sum()), 5))
        result[f"seed_tfidf_{dim}"] = scores
    
    return result

## 3. 전체 Feature Build 파이프라인

In [ ]:
import datetime

def build_features_v2(
    exp_id: str,
    tokenized_df: pd.DataFrame,
    output_dir: Path,
) -> pd.DataFrame:
    """
    tokenized_df → features parquet + 로그 CSV.
    
    5-key lineage 컬럼이 반드시 포함되어야 함.
    """
    print(f"\n{'='*50}")
    print(f"  Feature Build: {exp_id}  (N={len(tokenized_df)})")
    print(f"{'='*50}")
    
    # --- 2.1 Ratio features (row-wise) ---
    print("[1/3] ratio features...")
    ratio_rows = [compute_ratio_features(txt)
                  for txt in tokenized_df["joined_text"]]
    feat_df = pd.DataFrame(ratio_rows)
    
    # --- 2.2 TF-IDF concentration (corpus-level) ---
    print("[2/3] tfidf concentration...")
    documents = tokenized_df["joined_text"].tolist()
    feat_df["esg_tfidf_concentration"] = compute_tfidf_concentration(
        documents, seed_words=SEED_30, top_k=200
    )
    
    # --- 2.3 Seed TF-IDF E/S/G ---
    print("[3/3] seed tfidf E/S/G...")
    seed_scores = compute_seed_tfidf_by_dim(documents, seed_dict=SEED_DICT)
    for col, vals in seed_scores.items():
        feat_df[col] = vals
    
    # --- 5-key lineage 병합 ---
    id_cols_present = [c for c in ID_COLS if c in tokenized_df.columns]
    for col in id_cols_present:
        feat_df.insert(0, col, tokenized_df[col].values)
    
    feat_df["exp_id"] = exp_id
    
    # --- NaN 진단 ---
    n_nan = feat_df.isnull().sum().sum()
    print(f"\nTotal NaN cells: {n_nan}")
    print(feat_df.describe().T[["count","mean","std","min","max"]].round(4))
    
    # --- 저장 ---
    out_parquet = output_dir / f"features_{exp_id}.parquet"
    feat_df.to_parquet(out_parquet, index=False)
    print(f"\n→ saved: {out_parquet}")
    
    # --- 로그 ---
    log_entry = {
        "exp_id":           exp_id,
        "n_rows":           len(feat_df),
        "n_nan":            int(n_nan),
        "g_signal_ratio_mean":          feat_df["g_signal_ratio"].mean(),
        "esg_signal_ratio_mean":        feat_df["esg_signal_ratio"].mean(),
        "esg_tfidf_concentration_mean": feat_df["esg_tfidf_concentration"].mean(),
        "total_tokens_mean":            feat_df["total_tokens"].mean(),
        "run_ts":           datetime.datetime.now().isoformat(),
    }
    
    return feat_df, log_entry


# 전체 실행
feature_dfs = {}
log_rows = []

for exp_id, tdf in tokenized_dfs.items():
    feat_df, log_entry = build_features_v2(exp_id, tdf, D_FEATURES)
    feature_dfs[exp_id] = feat_df
    log_rows.append(log_entry)

# 로그 저장
log_df = pd.DataFrame(log_rows)
log_path = D_FEATURES / "feature_build_log.csv"
log_df.to_csv(log_path, index=False, encoding="utf-8-sig")
print(f"\n로그 저장: {log_path}")
log_df

## 4. KCGS 등급 병합 → merged_v_final.parquet

In [ ]:
def load_and_merge_kcgs(
    feat_df: pd.DataFrame,
    exp_id: str,
    output_dir: Path,
) -> pd.DataFrame:
    """
    features DataFrame에 KCGS 등급을 join.
    
    Join key: stock_code + fiscal_year
    (company_name으로 join 금지 — silent mismatch 위험)
    
    주의: KCGS 미평가 firm은 NaN 보존. 0으로 채우지 않는다.
    """
    # 기존 merged CSV 활용 (이미 KCGS join 완료)
    merged_path = D_MERGED / f"merged_{exp_id}.csv"
    
    if merged_path.exists():
        print(f"기존 merged CSV 로드: {merged_path}")
        merged = pd.read_csv(
            merged_path,
            dtype={"stock_code": str, "corp_code": str, "rcept_no": str}
        )
        merged["stock_code"] = merged["stock_code"].str.zfill(6)
    else:
        # KCGS 등급 직접 로드 후 join
        print("KCGS 등급 직접 로드...")
        kcgs_path = D_MERGED.parent / "kcgs_esg_ratings.csv"  # data/
        if not kcgs_path.exists():
            kcgs_path = D_MERGED.parent / "kcgs_grades_raw.csv"
        
        kcgs = pd.read_csv(kcgs_path, dtype={"stock_code": str})
        kcgs["stock_code"] = kcgs["stock_code"].str.zfill(6)
        
        feat_df_copy = feat_df.copy()
        feat_df_copy["stock_code"] = feat_df_copy["stock_code"].str.zfill(6)
        merged = feat_df_copy.merge(
            kcgs[["stock_code","esg_year","kcgs_grade"]],
            on=["stock_code","esg_year"],
            how="left",
        )
    
    # 등급 수치화
    if "kcgs_grade" in merged.columns:
        merged["kcgs_grade_7"] = merged["kcgs_grade"].map(GRADE_TO_7)
        merged["kcgs_grade_4"] = merged["kcgs_grade"].map(GRADE_TO_4)
        merged["kcgs_grade_high"] = (merged["kcgs_grade"].isin(["A","A+"])).astype(int)
    
    # feature 컬럼이 없으면 feat_df에서 가져오기
    ratio_cols = ["g_signal_ratio","esg_signal_ratio","esg_tfidf_concentration",
                  "bp_contamination_rate","total_tokens","log_tokens"]
    missing_ratio = [c for c in ratio_cols if c not in merged.columns]
    if missing_ratio:
        # feat_df와 merge (stock_code + fiscal_year + rcept_no 기준)
        join_keys = [c for c in ["stock_code","fiscal_year","rcept_no"] 
                     if c in merged.columns and c in feat_df.columns]
        add_cols = [c for c in ratio_cols + ["seed_tfidf_E","seed_tfidf_S","seed_tfidf_G"]
                    if c in feat_df.columns and c not in merged.columns]
        if add_cols:
            merged = merged.merge(feat_df[join_keys + add_cols], on=join_keys, how="left")
    
    # log_tokens 없으면 계산
    if "log_tokens" not in merged.columns and "total_tokens" in merged.columns:
        merged["log_tokens"] = np.log(merged["total_tokens"] + 1)
    
    print(f"\nMerged shape: {merged.shape}")
    if "kcgs_grade" in merged.columns:
        print("KCGS 분포:")
        print(merged["kcgs_grade"].value_counts().sort_index())
        print(f"\nNaN in kcgs_grade: {merged['kcgs_grade'].isna().sum()}")
    
    # 저장
    out_path = output_dir / f"merged_{exp_id}.parquet"
    merged.to_parquet(out_path, index=False)
    print(f"→ saved: {out_path}")
    
    return merged


merged_dfs = {}
for exp_id, feat_df in feature_dfs.items():
    merged_dfs[exp_id] = load_and_merge_kcgs(feat_df, exp_id, D_FEATURES)

# Primary 표본
df_primary = merged_dfs[PRIMARY_EXP].dropna(subset=["kcgs_grade_7"])
print(f"\n★ PRIMARY ({PRIMARY_EXP}) 회귀 표본: N={len(df_primary)}")

## 5. Feature 분포 요약 시각화

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm

# 한글 폰트 시도
try:
    plt.rcParams['font.family'] = 'Malgun Gothic'
except:
    plt.rcParams['font.family'] = 'DejaVu Sans'
plt.rcParams['axes.unicode_minus'] = False

df_plot = df_primary.copy()

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
fig.suptitle(f"Feature Distributions — {PRIMARY_EXP} (N={len(df_plot)})", fontsize=13)

feature_cols = [
    ("g_signal_ratio",            "G Signal Ratio"),
    ("esg_signal_ratio",          "ESG Signal Ratio"),
    ("esg_tfidf_concentration",   "ESG TF-IDF Concentration"),
    ("bp_contamination_rate",     "BP Contamination Rate"),
    ("total_tokens",              "Total Tokens"),
    ("kcgs_grade_7",              "KCGS Grade (7-scale)"),
]

for ax, (col, label) in zip(axes.flat, feature_cols):
    if col not in df_plot.columns:
        ax.text(0.5, 0.5, f"{col}\n없음", ha='center', va='center')
        ax.set_title(label)
        continue
    vals = df_plot[col].dropna()
    ax.hist(vals, bins=30, color='steelblue', edgecolor='white', alpha=0.85)
    ax.set_title(f"{label}\n(mean={vals.mean():.4f}, std={vals.std():.4f})")
    ax.set_xlabel(col)
    ax.set_ylabel("count")

plt.tight_layout()
fig_path = str(D_FEATURES / f"distributions_{PRIMARY_EXP}.png")
plt.savefig(fig_path, dpi=150, bbox_inches='tight')
plt.close()
print(f"→ figure saved: {fig_path}")

## 6. 실험별 Feature 비교 (robustness preview)

In [ ]:
# 실험별 주요 feature 통계 비교표
compare_rows = []
for exp_id, feat_df in feature_dfs.items():
    row = {"exp_id": exp_id, "N": len(feat_df)}
    for col in ["g_signal_ratio","esg_signal_ratio",
                "esg_tfidf_concentration","bp_contamination_rate","total_tokens"]:
        if col in feat_df.columns:
            row[f"{col}_mean"] = feat_df[col].mean()
            row[f"{col}_std"]  = feat_df[col].std()
    compare_rows.append(row)

compare_df = pd.DataFrame(compare_rows).round(5)
display(compare_df)

# CSV 저장
cmp_path = D_FEATURES / "feature_comparison_by_exp.csv"
compare_df.to_csv(cmp_path, index=False, encoding="utf-8-sig")
print(f"→ saved: {cmp_path}")

## 7. Merge Quality Report

KCGS merge 전후 N을 기록 — 왜 일부 firm-year가 제외되는지 명시

In [ ]:
quality_rows = []
for exp_id, mdf in merged_dfs.items():
    n_before = len(mdf)
    n_after  = int(mdf["kcgs_grade_7"].notna().sum()) if "kcgs_grade_7" in mdf.columns else None
    n_miss   = n_before - n_after if n_after is not None else None
    
    grade_dist = ""
    if "kcgs_grade" in mdf.columns:
        grade_dist = str(mdf["kcgs_grade"].value_counts().to_dict())
    
    quality_rows.append({
        "exp_id":       exp_id,
        "n_before_merge": n_before,
        "n_after_kcgs": n_after,
        "n_dropped":    n_miss,
        "drop_reason":  "KCGS 미평가 or 신규상장",
        "grade_dist":   grade_dist,
    })

qr_df = pd.DataFrame(quality_rows)
qr_path = D_FEATURES / "merge_quality_report.csv"
qr_df.to_csv(qr_path, index=False, encoding="utf-8-sig")
print("=== Merge Quality Report ===")
display(qr_df)
print(f"→ saved: {qr_path}")

## 8. Interpretation & Next Step

**이 단계에서 확인할 것:**
- `bp_contamination_rate`가 exp_E/F에서 0에 가까운가? → boilerplate filter 효과 확인
- `g_signal_ratio` 분포가 right-skewed? → log 변환 필요 여부
- total_tokens와 esg features 간 상관? → verbosity bias 진단 (다음 단계)

**다음 단계:** `04_feature_validation_v2.ipynb`
- Spearman ρ (feature × KCGS grade) + Bootstrap CI
- Mann-Whitney U test (high vs low grade)
- VIF 사전 진단